In [1]:
from pathlib import Path
from typing import Tuple, List
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from __future__ import annotations

In [2]:
class Capability:
    def __init__(self, line, parent):
        #print(line)
        _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms   = line.split()
        self.boundsOffset = int(boundsOffset, 16)
        self.boundsLength = int(boundsLength, 16)
        self.boundsVirtBase = int(boundsVirtualBase, 16)
        self.capPerms = int(capPerms, 16)
        self.parent = parent
        parent.child = self

    def capKey(self):
        return (self.boundsVirtBase, self.boundsLength)
    
    def __eq__(self, other):
        return self.boundsOffset == other.boundsOffset and self.boundsLength == other.boundsLength and self.boundsVirtBase == other.boundsVirtBase and self.capPerms == other.capPerms

    def __hash__(self):
        return hash((self.boundsOffset, self.boundsLength, self.boundsVirtBase, self.capPerms))
    
    def __repr__(self):
        return f"Cap virtBase {self.boundsVirtBase} length {self.boundsLength} offset {self.boundsOffset}"

class ReportAccessLog:
    def __init__(self, line, index):
        #44200 Prefetcher logReportAccess level 1 addr 00000000c0007648 pcHash c0000260 hitMiss 1 boundsOffset 0000000000000008 boundsLength 0000000000002f30 boundsVirtBase 00000000c0007640 capPerms 0000000000001111000111111010111

        #print(line.split())
        clock_time,_, _, _, level, _, addr, _, pcHash, _, isMiss, _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms, _, op   = line.split()
        self.clock_time = int(clock_time)/10
        self.level = int(level)
        self.addr = int(addr, 16)
        self.pcHash = int(pcHash, 16)
        self.isMiss = int(isMiss)
        self.op = op
        self.cap = Capability("boundsOffset"+line.partition("op")[0].partition("boundsOffset")[2], self)

        self.index = index
    
class ReportDataArrivalCap:
    def __init__(self, line, parent):
        pre, _, cap = line.partition("boundsOffset")
        _, _, _, _, index, _, tag, _, addr = pre.split()
        self.index = int(index)
        self.tag = bool(int(tag))
        self.addr = int(addr, 16)
        if self.tag:
            self.cap = Capability("boundsOffset"+cap, parent)
        self.parent = parent
        parent.child = self

class ReportDataArrival:
    def __init__(self, line, index):
        clock_time,_, _, _, level, _, requestAddr, _, pcHash, _, wasMiss, _, wasPrefetch, _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms, _, op  = line.split()
        self.clock_time = int(clock_time)/10
        self.level = int(level)
        self.requestAddr = int(requestAddr, 16)
        self.pcHash = int(pcHash, 16)
        self.wasMiss = int(wasMiss)
        self.wasPrefetch = int(wasPrefetch)

        #print(line)
        self.requestCap = Capability("boundsOffset"+line.partition("op")[0].partition("boundsOffset")[2], self)
        self.capabilites: List[ReportDataArrivalCap] = []

        self.sel_capability = None

        self.index = index

    def add_capability(self, line, parent_data_arrivals):
        capability = ReportDataArrivalCap(line, self)
        if capability.tag:
            self.capabilites.append(capability)
            parent_data_arrivals[capability.cap.capKey()] = capability.cap 

        return capability

    def add_sel_capability(self, line):
        capability = ReportDataArrivalCap(line, self)
        if capability.tag:
            self.sel_capability = capability 

        return capability




def parse_log(input_file) -> Tuple[List[ReportAccessLog], List[ReportDataArrival], List[ReportAccessLog]]:
    total_order_events = []
    recent_misses = set()
    parent_access = {} #(cap_virtual_base, cap_size, most recent ReportAccessLog)
    parent_data_arrivals = {} #(cap_virtual_base, cap_size, most recent ReportAccessLog)
    accesses = []
    data_arrivals = []
    access_misses = []

    access_index = 0
    data_arrival_index = 0
    

    with open(input_file, "r") as fp:
        while True:
            line = fp.readline()
            if not line:
                break

            if "logReportAccess" in line:
                reportAccess = ReportAccessLog(line, access_index)
                total_order_events.append(reportAccess)


                access_index += 1

                if reportAccess.isMiss:
                    recent_misses.add(reportAccess.addr)
                    access_misses.append(reportAccess)
                elif reportAccess.addr in recent_misses:
                    #Skip first hit access after miss
                    recent_misses.remove(reportAccess.addr)
                    continue

                accesses.append(reportAccess)
                parent_access[reportAccess.cap.capKey()] = reportAccess.cap

                if reportAccess.cap.capKey() in parent_data_arrivals:
                    parent = parent_data_arrivals[reportAccess.cap.capKey()]
                    reportAccess.parent = parent
                    parent.child = reportAccess
                    #parent.child = reportAccess
                    #print("found parent")

                # if reportAccess.cap.boundsLength == 40000:
                #         print("40000 arrival")
                #         print(reportAccess.cap.capKey())
                #print(reportAccess.cap.boundsLength)
            
            elif "logReportDataArrival" in line:
                data_arrival = ReportDataArrival(line, data_arrival_index)
                total_order_events.append(data_arrival)
                data_arrival_index += 1 

                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)

                data_arrival.add_sel_capability(fp.readline())

                data_arrivals.append(data_arrival)

                if data_arrival.requestCap.capKey() in parent_access:
                    # if data_arrival.requestCap.boundsLength == 40000:
                    #     print("40000 arrival in arrival")
                    parent = parent_access[data_arrival.requestCap.capKey()]
                    data_arrival.parent = parent
                    parent.child = data_arrival
                    #parent.child = data_arrival
                    #print("found parent2")

                # if data_arrival.requestCap.boundsLength == 40000:
                #     print(data_arrival.requestCap.capKey())
                    


    print(f"access {len(accesses)} misses {len(access_misses)}")
    return accesses, data_arrivals, access_misses, total_order_events

In [3]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("../simulations/patricia") / "sim_0.0" / "sim_stdout"

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file)

access 735380 misses 69290


In [37]:
from collections import deque
from functools import partial


class BackwardTableEntry:
    def __init__(self, parentVirtBase, parentOffset, parentTime):
        self.parentVirtBase = parentVirtBase
        self.parentOffset = parentOffset
        self.parentTime = parentTime

class TimelinessTableEntry:
    def __init__(self, pcHash, clock_time):
        self.pcHash = pcHash
        self.clockTime = clock_time
        
class PredictionTableEntry:
    def __init__(self, parentOffset, childOffset, confidence, pcHash):
        self.parentOffset = parentOffset
        self.childOffset = childOffset
        self.confidence = confidence
        self.pcHash = pcHash

class PrefetchRequest:
    def __init__(self, addr, time, childOffset, predictionEntry):
        self.addr = addr
        self.time = time
        self.childOffset = childOffset
        self.predictionEntry = predictionEntry

class BaseBackwardsPrefetcher:
    def __init__(self, timelinessWays, predictionWays, predictionReplacementConfidence, predictionPrefetchConfidence):
        self.backwardsTable = {}
        self.timelinessTable = defaultdict(partial(deque, maxlen=timelinessWays))
        self.predictionTable = defaultdict(list)

        self.predictionWays = predictionWays
        self.predictionReplacmentConfidence = predictionReplacementConfidence
        self.predictionPrefetchConfidence = predictionPrefetchConfidence

        self.totalCacheMisses = 0
        self.prefetchesNotMisses = 0
        self.prefetchesMisses = 0
        
        self.prefetches = []
        self.successfulPrefetches = []

    def addToPredictionTable(self, pcHash, parentOffset, childOffset):
        ways = self.predictionTable[pcHash]
        
        # if len(ways)==0:
        #     self.predictionTable[pcHash] = [PredictionTableEntry(parentOffset, childOffset, 1)]
        #     return
        
        lowestConfidence = 100000000000000
        lowestConfidenceIndex = -1

        way: PredictionTableEntry
        for i, way in enumerate(ways):
            if way.parentOffset == parentOffset and way.childOffset == childOffset:
                way.confidence += 1
                ways[i] = way
                self.predictionTable[pcHash] = ways
                return

            if way.confidence < lowestConfidence:
                lowestConfidence = way.confidence
                lowestConfidenceIndex = i
        
        if len(ways) < self.predictionWays:
            ways.append(PredictionTableEntry(parentOffset, childOffset, 1, pcHash))
            self.predictionTable[pcHash] = ways
            return


        # Replace if below confidence 
        if lowestConfidence < self.predictionReplacmentConfidence:
            ways[lowestConfidenceIndex] = PredictionTableEntry(parentOffset, childOffset, 1, pcHash)
        else:
            # Otherwise decrease confidence for all other ways (is all the correct approach)
            for i, way in enumerate(ways):
                way.confidence -= 1
                ways[lowestConfidenceIndex] = way
        
        self.predictionTable[pcHash] = ways

    def reportRequest(self, report_access: ReportAccessLog):
        if report_access.isMiss:
            self.totalCacheMisses += 1
            if report_access.cap.boundsVirtBase in self.backwardsTable:
                parentVirtBase = self.backwardsTable[report_access.cap.boundsVirtBase].parentVirtBase
                parentOffset = self.backwardsTable[report_access.cap.boundsVirtBase].parentOffset
                
                timeliness_queue = self.timelinessTable[parentVirtBase]
                if len(timeliness_queue) > 0:
                   # print(f"timeliness queue length {len(timeliness_queue)}")
                    pcToPrefetchOn = timeliness_queue[0].pcHash # getting oldest value for simple prefetcher

                    # # Should prefetch table have multiple ways - answer is probably yes but confidence may be enough 
                    # if pcToPrefetchOn in self.predictionTable:
                    #     if self.predictionTable[pcToPrefetchOn].confidence <= 0:
                    #         self.predictionTable[pcToPrefetchOn] = PredictionTableEntry(parentOffset, report_access.cap.boundsOffset, 1)
                    #     else:
                    #         currentPrediction = self.predictionTable[pcToPrefetchOn]
                    #         if currentPrediction.parentOffset == parentOffset and currentPrediction.childOffset == report_access.cap.boundsOffset:
                    #             # TODO: do we want to increase confidence on seeing same miss twice? not sure we do
                    #             currentPrediction.confidence += 1
                    #             self.predictionTable[pcToPrefetchOn] = currentPrediction
                    #             pass
                    #         else:
                    #             # Decrease condfidence if different prefetch pattern seen from PC (definitely possible and may want multiple ways)
                    #             currentPrediction.confidence -= 1
                    #             self.predictionTable[pcToPrefetchOn] = currentPrediction
                    # else:
                    #     self.predictionTable[pcToPrefetchOn] = PredictionTableEntry(parentOffset, report_access.cap.boundsOffset, 1)

                    self.addToPredictionTable(pcToPrefetchOn, parentOffset, report_access.cap.boundsOffset)


        self.timelinessTable[report_access.cap.boundsVirtBase].append(TimelinessTableEntry(report_access.pcHash, report_access.clock_time))


        way: PredictionTableEntry
        for way in self.predictionTable[report_access.pcHash]:
            if way.confidence >= self.predictionPrefetchConfidence:
                self.prefetches.append(PrefetchRequest(report_access.cap.boundsVirtBase + way.parentOffset, report_access.clock_time, way.childOffset, way))
            
        
    def reportDataArrival(self, data_arrival: ReportDataArrival):
        # Build backwards table
        # TODO: should I check whether it is sub capability?
        if data_arrival.sel_capability and data_arrival.requestCap.boundsVirtBase != data_arrival.sel_capability.cap.boundsVirtBase:
            self.backwardsTable[data_arrival.sel_capability.cap.boundsVirtBase] = BackwardTableEntry(data_arrival.requestCap.boundsVirtBase, 
                                                                                                     data_arrival.requestCap.boundsOffset, 
                                                                                                     data_arrival.clock_time)
        
        prefetchHit = None
        for prefetch in self.prefetches:
            #print(f"{prefetch.addr} {data_arrival.requestAddr}")
            if prefetch.addr == data_arrival.requestAddr:
                prefetchHit = True
                if prefetch.childOffset and data_arrival.sel_capability:
                    self.prefetches.append(PrefetchRequest(data_arrival.sel_capability.cap.boundsVirtBase + prefetch.childOffset, data_arrival.clock_time, None, prefetch.predictionEntry))
                prefetchHit = prefetch
                self.prefetches.remove(prefetch)
        
        # if prefetchHit == True:        
        #     print(f"Prefetch match wasMiss {data_arrival.wasMiss}")
        
        if prefetchHit and data_arrival.wasMiss:
            self.prefetchesMisses += 1

            ways = self.predictionTable[prefetchHit.predictionEntry.pcHash]
            way: PredictionTableEntry
            for i, way in enumerate(ways):
                if way.parentOffset == prefetchHit.predictionEntry.parentOffset and way.childOffset == prefetchHit.predictionEntry.childOffset:
                    way.confidence += 1
                    ways[i] = way
                else:
                    way.confidence -= 1
                    ways[i] = way
            self.predictionTable[prefetchHit.predictionEntry.pcHash] = ways
                    
        # if not prefetchHit and data_arrival.wasMiss:
        #     ways = self.predictionTable[prefetchHit.predictionEntry.pcHash]
        #     way: PredictionTableEntry
        #     for i, way in enumerate(ways):
        #         way.confidence -= 1
        #         ways[i] = way
        #     self.predictionTable[prefetchHit.predictionEntry.pcHash] = ways

        if prefetchHit and not data_arrival.wasMiss:
            self.prefetchesNotMisses += 1

        # On miss 
    def reportStatistics(self):
        return self.totalCacheMisses, self.prefetchesNotMisses, self.prefetchesMisses, len(self.prefetches)

In [43]:
prefetcher = BaseBackwardsPrefetcher(8, 1, 1, 4)
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(f"PredConfidence {2} stats: {prefetcher.reportStatistics()}")

PredConfidence 2 stats: (10912, 19453, 3036, 1033)


In [17]:
prefetcher.predictionTable

defaultdict(list,
            {3221225666: [],
             3221225862: [],
             3221225864: [],
             3221225880: [],
             3221226096: [],
             3221226102: [],
             3221226064: [],
             3221226076: [],
             3221226168: [],
             3221226178: [],
             3221226088: [<__main__.PredictionTableEntry at 0x7f72564aaac0>],
             3221226274: [],
             3221233244: [],
             3221233248: [],
             3221233252: [],
             3221233256: [],
             3221226304: [],
             3221226316: [],
             3221233260: [],
             3221233264: [],
             3221226308: [<__main__.PredictionTableEntry at 0x7f72f9bf68b0>],
             3221226320: [],
             3221233280: [],
             3221226336: [],
             3221227392: [],
             3221227396: [],
             3221227400: [],
             3221227404: [],
             3221227408: [],
             3221227412: [],
             3

In [8]:
prefetcher = BaseBackwardsPrefetcher(8)
for event in total_order_events[:100]:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(prefetcher.reportStatistics())

(69290, 244751, 37649, 97852)


In [ ]:
prefetcher.predictionTable.keys[0]

In [ ]:
37649/37649+97852

In [ ]:
# Building table to work backwards from
backwardsFromMiss = {}

offsets=set()
for data_arrival in data_arrivals:
    if data_arrival.sel_capability:
        print(data_arrival.sel_capability)
    # backwardsFromMiss[ba]

In [45]:
offsets

{'0xc000',
 '0xc001',
 '0xc002',
 '0xc003',
 '0xc004',
 '0xc005',
 '0xc006',
 '0xc007',
 '0xc008',
 '0xc009',
 '0xc00a',
 '0xc00b',
 '0xc00c',
 '0xc00d',
 '0xc00e',
 '0xc00f',
 '0xc010',
 '0xc011',
 '0xc012',
 '0xc013',
 '0xc014',
 '0xc015',
 '0xc016',
 '0xc017',
 '0xc018',
 '0xc019',
 '0xc01a',
 '0xc01b',
 '0xc01c',
 '0xc01d',
 '0xc01e',
 '0xc01f',
 '0xc020',
 '0xc021',
 '0xc022'}